# Label Analyzer - Production CLP Compliance Checker

**Two-Layer Deterministic Validation System**

## Architecture

**Stage 0: DPI Calibration**
- Auto-detects measurement lines on label
- Calculates true DPI for accurate font size measurements

**Stage 1: Region Detection**
- Identifies CLP (regulatory) vs Non-CLP (marketing) sections
- Content types: Ingredients, Hazard Symbols, Warnings, etc.

**Stage 2: Boundary Refinement**
- Refines boundaries and detects irregular shapes

**Stage 3: CLP Compliance Measurement & Validation**
- **Layer 1 (Gemini)**: Measures font size (mm), line distance (mm), contrast
- **Layer 2 (Local)**: Applies deterministic rules (100% reproducible)

**Stage 4: Filtering & Confidence Thresholding**
- Flags borderline/uncertain results for human review


## Setup

In [ ]:
!pip install -q google-genai pillow pymupdf pydantic

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from label_analyzer_production import LabelAnalyzer, PartClassification, image_to_base64, pdf_to_image
from PIL import Image as PIL_Image
import json, os
from datetime import datetime

print('✅ Imports OK')

In [ ]:
PROJECT_ID = 'your-gcp-project-id'
DPI = 300
PACKAGE_SIZE_ML = 500
os.makedirs('data/in', exist_ok=True)
os.makedirs('data/out', exist_ok=True)
print(f'✅ Config: {PROJECT_ID}, {DPI} DPI')

## Load Label Image

In [ ]:
IMAGE_PATH = 'data/in/your_label.pdf'
if IMAGE_PATH.lower().endswith('.pdf'):
    img = pdf_to_image(IMAGE_PATH, dpi=DPI)
else:
    img = PIL_Image.open(IMAGE_PATH)
print(f'✅ Loaded: {img.size[0]}×{img.size[1]} px')
display(img.resize((img.width // 3, img.height // 3)))

## Analyze

In [ ]:
!gcloud auth application-default login

In [ ]:
analyzer = LabelAnalyzer(project_id=PROJECT_ID, dpi=DPI)
image_data = image_to_base64(img)
print('⏳ Analyzing...')
parts = analyzer.analyze(img, image_data)
print(f'✅ {len(parts)} regions detected')

## Results

In [ ]:
clp = [p for p in parts if p.classification == PartClassification.CLP]
compliant = [p for p in clp if p.is_compliant()]
review = [p for p in clp if p.needs_human_review()]

print('='*80)
print(f'CLP Regions: {len(clp)} | Compliant: {len(compliant)} | Review: {len(review)}')
print('='*80)
print(f'{'Region':<30} {'Status':<15} {'Confidence':<10}')
for p in clp:
    s = 'PASS ✓' if p.is_compliant() else 'FAIL ✗'
    if p.needs_human_review():
        s += ' (REVIEW)'
    c = p.compliance_check.get('measurement_confidence', 0) if p.compliance_check else 0
    print(f'{p.label:<30} {s:<15} {c:.0%}')

## Detailed Rule Analysis

In [ ]:
for i, p in enumerate(clp, 1):
    if not p.compliance_check:
        continue
    print(f'\n{'='*80}')
    print(f'Region {i}: {p.label} ({p.content_type})')
    m = p.compliance_check.get('measurements', {})
    print(f'Font: {m.get(\"font_size_mm\", 0):.2f}mm | Line: {m.get(\"line_distance_mm\", 0):.2f}mm | Conf: {m.get(\"measurement_confidence\", 0):.0%}')
    r = p.compliance_check.get('rule_results', {})
    r1 = r.get('rule_1_font_size', {})
    r2 = r.get('rule_2_line_distance', {})
    r3 = r.get('rule_3_background_contrast', {})
    print(f'Font: {r1.get(\"status\")} | Line: {r2.get(\"status\")} | Contrast: {r3.get(\"status\")} → {p.compliance_check.get(\"overall_compliance\")}')

## Visualization

In [ ]:
viz = analyzer.visualize(img)
display(viz.resize((viz.width // 3, viz.height // 3)))
viz.save('data/out/labeled.jpg')
print('✅ Saved: data/out/labeled.jpg')

## Export JSON

In [ ]:
report = {'timestamp': datetime.now().isoformat(), 'image': IMAGE_PATH, 'regions': []}
for p in parts:
    report['regions'].append({
        'type': p.classification.value,
        'label': p.label,
        'content': p.content_type,
        'compliant': p.is_compliant(),
        'review': p.needs_human_review(),
        'compliance': p.compliance_check
    })
with open('data/out/report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(f'✅ Saved: data/out/report.json')